# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
This notebook demonstrates step-by-step processing and exploration of the FAIR⁲ dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading

Load the dataset schema and core metadata using `mlcroissant`. This ensures reproducibility and easy referencing by entity `@id` in the following analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get the simple metadata and print info
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview

Review all available record sets (tables), and for each, show the accessible field `@id`s (columns, attributes) and their descriptions.

In [ ]:
# List all record sets and their fields/columns by @id
print("Available record sets and their fields:")
for rs in dataset.record_sets:
    print(f"\nRecord Set @id: {rs.id}")
    print(f"  Title/name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields (by @id):")
    for fld in rs.fields:
        info = f"    - @id: {fld.id} | Name: {fld.name if hasattr(fld, 'name') else ''}"
        if hasattr(fld, "description") and fld.description:
            info += f" | Desc: {fld.description}" 
        print(info)

## 3. Data Extraction

This section loads all records for a chosen record set and inspects its columns. We refer to record sets and fields by their `@id` as instructed for reproducibility.

**Choose which record set(s) to load by updating the `record_set_ids` variable below (copied from output above).**

In [ ]:
# List all available record set @id for easy reference
record_sets_ids = [rs.id for rs in dataset.record_sets]
print("All record set @ids:", record_sets_ids)

# For this dataset, the main record set likely looks like: 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
# or similar - choose main one based on overview above.
# Update the record set(s) list as desired:
main_record_set_id = record_sets_ids[0] if record_sets_ids else None  # Use the first one
record_set_ids = [main_record_set_id]

# Load all records for each record set into a DataFrame, use the @id as a key
dfs = {}
for rs_id in record_set_ids:
    if rs_id:
        print(f"\nLoading records for record set @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        dfs[rs_id] = pd.DataFrame(records)

# Show the columns of the main loaded DataFrame
if main_record_set_id:
    print(f"\nAvailable columns (@id) for record set {main_record_set_id}:")
    print(dfs[main_record_set_id].columns.tolist())
    display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Demonstrate basic EDA operations using field `@id`s referring to selected numeric and group fields.

- **Numeric Field Example:** Select a field (by `@id`) such as 'cr:molecularWeight', 'cr:peptideCount', or any other numeric field available in the data.
- **Group Field Example:** Select a suitable grouping variable, e.g., 'cr:sample' or 'cr:condition', as available.


In [ ]:
df = dfs[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()
# Print available columns to choose valid @ids for fields
print('DataFrame columns:', df.columns.tolist())

# ------- Adjust field @id variables below to select numerical and grouping fields based on printed columns --------
numeric_field = None
# Examples: 'cr:molecularWeight', 'cr:peptideCount', 'cr:normalizedAbundance' (replace with actual @id from data)
possible_numeric = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[c])]
if possible_numeric:
    numeric_field = possible_numeric[0]
else:
    # Try to convert columns that look numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            continue
    # Retry selection
    possible_numeric = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[c])]
    if possible_numeric:
        numeric_field = possible_numeric[0]

print(f"Selected numeric field: {numeric_field}")

# Filtering records
if numeric_field:
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
    print(f"\nNormalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find a suitable group field
    group_field = None
    for col in df.columns:
        if df[col].nunique() < 10 and col != numeric_field:
            group_field = col
            break
    print(f"Suggested group_field: {group_field}")
    
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field detected for EDA.")

## 5. Visualization

Plot the distribution of the chosen numeric field, and (if possible) compare means by a group field.

Examples: histogram of molecular weight; boxplot for abundance by condition.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, you explored the FAIR⁲ mass spectrometry dataset using `mlcroissant`:
+ Loaded full Croissant metadata and inspected record sets and fields using their `@id`.
+ Loaded records from the main record set and performed basic filtering and normalization using field `@id`s.
+ Visualized numeric distributions and reviewed summary statistics per group.

**Next steps:** Apply further domain-specific filtering, integrate with external proteomics resources, or use the processed DataFrame for downstream analysis or ML workflows.

For advanced use, see [`mlcroissant` documentation](https://github.com/mlcommons/croissant) and your dataset's full Croissant schema.